In [1]:
from pathlib import Path
_template = str(Path(__vsc_ipynb_file__).parent.parent / '../template.ipynb')
%run "$_template"

In [2]:
import lightgbm as lgb

In [3]:
df_test = pd.read_csv("../../data/modeling/test.csv")

photo_softbake = [
    'photo_soft_Chamber',
    'resist_target',
    'N2_HMDS',
    'pressure_HMDS',
    'temp_HMDS',
    'temp_HMDS_bake',
    'time_HMDS_bake',
    'spin1',
    'spin2',
    'spin3',
    'photoresist_bake',
    'temp_softbake',
    'time_softbake'
]
X = df_test[photo_softbake].copy()
y = df_test['is_low_yield'].copy()

In [10]:
# =========================
# 1. 모델 불러오기
# =========================
photo_softbake_model = lgb.Booster(model_file="../../model/photo_softbake_lgbm.txt")

# =========================
# 2. 불량 예측 확률 계산
# =========================
bad_prob = photo_softbake_model.predict(X)

result = X.copy()
result["bad_prob"] = bad_prob
result["y_true"] = y.values if hasattr(y, "values") else y

# =========================
# 3. 위험구간 기준 설정
# =========================
low_threshold = 0.1
high_ratio = 0.1

high_threshold = result["bad_prob"].quantile(1 - high_ratio)

print("저위험 기준 bad_prob <=", low_threshold)
print("고위험 기준 bad_prob >=", high_threshold)

# =========================
# 4. 위험구간 부여
# =========================
def assign_risk_group(prob):
    if prob <= low_threshold:
        return "저위험"
    elif prob >= high_threshold:
        return "고위험"
    else:
        return "중위험"

result["risk_group"] = result["bad_prob"].apply(assign_risk_group)

# =========================
# 5. 위험구간별 성능 요약
# =========================
risk_summary = (
    result
    .groupby("risk_group")
    .agg(
        data_count=("y_true", "count"),
        actual_defect_count=("y_true", "sum"),
        actual_defect_rate=("y_true", "mean"),
        mean_pred_prob=("bad_prob", "mean"),
        min_pred_prob=("bad_prob", "min"),
        max_pred_prob=("bad_prob", "max")
    )
    .reset_index()
)

risk_summary["data_ratio_percent"] = risk_summary["data_count"] / len(result) * 100
risk_summary["actual_defect_rate_percent"] = risk_summary["actual_defect_rate"] * 100
risk_summary["mean_pred_prob_percent"] = risk_summary["mean_pred_prob"] * 100

display(risk_summary)

# =========================
# 6. 결과 확인
# =========================
display(result[["bad_prob", "y_true", "risk_group"]].head())

저위험 기준 bad_prob <= 0.1
고위험 기준 bad_prob >= 0.9907839101091358


,risk_group,data_count,actual_defect_count,actual_defect_rate,mean_pred_prob,min_pred_prob,max_pred_prob,data_ratio_percent,actual_defect_rate_percent,mean_pred_prob_percent
0,고위험,462,462,1.000000,0.998665,9.908045e-01,0.999998,10.006498,100.000000,99.866458
1,저위험,4021,4,0.000995,0.001102,6.295764e-10,0.097044,87.091185,0.099478,0.110244
2,중위험,134,118,0.880597,0.796502,1.061729e-01,0.990770,2.902318,88.059701,79.650190


,bad_prob,y_true,risk_group
0,0.003146,0,저위험
1,0.000060,0,저위험
2,0.999865,1,고위험
3,0.001789,0,저위험
4,0.000715,0,저위험
